In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# ======================
# МОНТИРОВАНИЕ GOOGLE DRIVE
# ======================
print("Монтирование Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive успешно смонтирован!")

# ======================
# СОЗДАНИЕ ДИРЕКТОРИИ ДЛЯ РЕЗУЛЬТАТОВ НА GOOGLE DRIVE
# ======================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_results_dir = "/content/drive/MyDrive/fish_weight_prediction_results"
output_results_dir = os.path.join(drive_results_dir, f"analysis_{timestamp}")

# Создаем основную папку и подпапку с timestamp
os.makedirs(drive_results_dir, exist_ok=True)
os.makedirs(output_results_dir, exist_ok=True)

print(f"Основная папка для результатов: {drive_results_dir}")
print(f"Папка для текущего анализа: {output_results_dir}")

# ======================
# DATA LOADING FROM LOCAL EXCEL FILE ON GOOGLE DRIVE
# ======================
try:
    # Загрузка данных из локального Excel файла на Google Drive
    file_path = r"/content/drive/MyDrive/Общая_база_карпы_22_05,_23_05_с_возрастом.xlsx"

    # Проверяем существование файла
    if not os.path.exists(file_path):
        print(f"Файл не найден: {file_path}")
        print("Ищем файл в других местах...")

        # Попробуем найти файл в родительских директориях
        possible_paths = [
            "/content/drive/MyDrive/02-02-26-Общая база (22-05, 23-05).xlsx",
            "/content/drive/MyDrive/Данные/02-02-26-Общая база (22-05, 23-05).xlsx",
            "/content/drive/MyDrive/Проекты/02-02-26-Общая база (22-05, 23-05).xlsx",
            "/content/drive/MyDrive/Работа/02-02-26-Общая база (22-05, 23-05).xlsx"
        ]

        found = False
        for path in possible_paths:
            if os.path.exists(path):
                file_path = path
                print(f"Файл найден: {file_path}")
                found = True
                break

        if not found:
            print("Файл не найден в стандартных местах. Пожалуйста, проверьте путь.")
            print("Создаем тестовые данные...")
            raise FileNotFoundError(f"Файл не найден")
        else:
            # If found, we proceed; if not found, the previous 'raise FileNotFoundError' handles it.
            pass

    print(f"Загрузка данных из файла: {file_path}")

    # Загружаем Excel файл
    excel_file = pd.ExcelFile(file_path)
    print(f"Доступные листы: {excel_file.sheet_names}")

    # Пробуем загрузить все листы для анализа
    sheet_data = {}
    for sheet_name in excel_file.sheet_names:
        try:
            df_temp = pd.read_excel(file_path, sheet_name=sheet_name)
            sheet_data[sheet_name] = df_temp
            print(f"  Лист '{sheet_name}': {df_temp.shape[0]} строк, {df_temp.shape[1]} колонок")
        except Exception as e:
            print(f"  Ошибка при загрузке листа '{sheet_name}': {e}")

    # Загружаем первый лист как основной датасет
    if sheet_data:
        first_sheet = list(sheet_data.keys())[0]
        df = sheet_data[first_sheet]
        print(f'Основные данные загружены из листа: "{first_sheet}"')

        # Store the initial raw DataFrame before any modifications
        df_raw_initial = df.copy()

        # Сохраняем информацию о всех листах в файл
        sheets_info_path = os.path.join(output_results_dir, f"excel_sheets_info_{timestamp}.txt")
        with open(sheets_info_path, 'w', encoding='utf-8') as f:
            f.write("Информация о листах Excel файла:\n")
            f.write("="*50 + "\n")
            for sheet_name, df_temp in sheet_data.items():
                f.write(f"\nЛист: {sheet_name}\n")
                f.write(f"Размер: {df_temp.shape[0]} строк, {df_temp.shape[1]} колонок\n")
                f.write(f"Колонки: {list(df_temp.columns)}\n")
        print(f"\u2713 Информация о листах сохранена: {sheets_info_path}")
    else:
        raise Exception("Не удалось загрузить ни один лист из Excel файла")

    print('Данные успешно загружены из локального Excel файла!')

except FileNotFoundError as e:
    print(f"Ошибка: {e}")
    print("Попытка загрузить тестовые данные...")
    # Создаем тестовые данные для демонстрации
    np.random.seed(42)
    n_samples = 200
    data = {
        'Вес, г': np.random.uniform(100, 5000, n_samples),
        'Длина, см': np.random.uniform(10, 80, n_samples),
        'Промыс.длина, см': np.random.uniform(12, 85, n_samples),
        'Обхват, см': np.random.uniform(8, 40, n_samples),
        'Наиб. Высота, см': np.random.uniform(5, 25, n_samples),
        'Вид': np.random.choice(['Кета', 'Горбуша', 'Нерка', 'Чавыча', 'Кижуч'], n_samples),
        'Возраст': np.random.randint(1, 11, n_samples) # Added 'Возраст' column
    }
    df = pd.DataFrame(data)
    print('Созданы тестовые данные!')

    # Store the initial raw DataFrame if test data is created
    df_raw_initial = df.copy()

    # Сохраняем тестовые данные в Google Drive
    test_data_path = os.path.join(output_results_dir, f"test_data_created_{timestamp}.xlsx")
    df.to_excel(test_data_path, index=False)
    print(f"\u2713 Тестовые данные сохранены: {test_data_path}")

except Exception as e:
    print(f"Ошибка при загрузке Excel файла: {e}")
    raise

# Сохраняем информацию об исходном файле
file_info_path = os.path.join(output_results_dir, f"source_file_info_{timestamp}.txt")
with open(file_info_path, 'w', encoding='utf-8') as f:
    f.write(f"Исходный файл: {file_path}\n")
    f.write(f"Дата загрузки: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Размер файла: {os.path.getsize(file_path) if os.path.exists(file_path) else 'N/A'} байт\n")

# Показываем информацию о данных
print('\nПервые строки данных:')
print(df.head())
print('\nИнформация о данных:')
print(df.info())
print(f'\nРазмер данных: {df.shape}')
print(f'\nКолонки в данных: {df.columns.tolist()}')

# Сохраняем информацию о данных в файл
data_info_path = os.path.join(output_results_dir, f"data_info_{timestamp}.txt")
with open(data_info_path, 'w', encoding='utf-8') as f:
    f.write("Информация о данных:\n")
    f.write("="*50 + "\n")
    f.write(f"Размер данных: {df.shape}\n")
    f.write(f"Колонки: {df.columns.tolist()}\n\n")
    f.write("Типы данных:\n")
    f.write(str(df.dtypes))
    f.write("\n\nОписательная статистика:\n")
    f.write(str(df.describe()))

# Проверяем и переименовываем колонки для удобства
column_mapping = {
    'Вес, г': 'Weight',
    'Длина, см': 'Length',
    'Промыс.длина, см': 'Commercial_Length',
    'Обхват, см': 'Girth',
    'Наиб. Высота, см': 'Max_Height',
    'Вид': 'Species',
    'Возраст': 'Age',
    'Возраст, л (по состоянию на весну 2025)': 'Age' # Added explicit mapping for the actual column name
}

# Создаем обратное отображение для русских названий
reverse_mapping = {v: k for k, v in column_mapping.items()}

# Переименовываем колонки, если они существуют
renamed_columns = {}
for rus_name, eng_name in column_mapping.items():
    if rus_name in df.columns:
        df.rename(columns={rus_name: eng_name}, inplace=True)
        renamed_columns[rus_name] = eng_name
        print(f"Переименовано: '{rus_name}' -> '{eng_name}'")

print(f'\nКолонки после переименования: {df.columns.tolist()}')

# Сохраняем информацию о переименовании
rename_info_path = os.path.join(output_results_dir, f"column_renaming_{timestamp}.txt")
with open(rename_info_path, 'w', encoding='utf-8') as f:
    f.write("Переименование колонок:\n")
    f.write("="*50 + "\n")
    for rus, eng in renamed_columns.items():
        f.write(f"{rus} -> {eng}\n")

# ======================
# DATA PREPROCESSING
# ======================
def preprocess_data(df):
    """Clean and prepare the dataset for modeling"""

    # Создаем копию данных
    df = df.copy()

    # --- Age handling START ---
    # The 'Age' column should already be renamed due to the updated column_mapping.
    # Now just ensure it's numeric and handle NaNs if any.
    if 'Age' not in df.columns:
        print("   Создаем синтетический признак 'Age' (отсутствует колонка)...")
        df['Age'] = np.random.randint(1, 11, df.shape[0])
    if 'Age' in df.columns:
        df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
        print("   Колонка 'Age' приведена к числовому типу.")
    # --- Age handling END ---

    # 1. Проверяем наличие необходимых колонок (before dropping others)
    required_columns = ['Weight', 'Length', 'Girth', 'Max_Height', 'Age']
    if 'Species' in df.columns: # Species might be present depending on data
        required_columns.append('Species')
    if 'Commercial_Length' in df.columns: # Commercial_Length might be present
        required_columns.append('Commercial_Length')

    print("\nПроверка необходимых колонок:")

    available_columns = []
    for col in required_columns:
        if col in df.columns:
            available_columns.append(col)
            print(f"\u2713 {col} доступна")
        else:
            print(f"\u2717 {col} отсутствует")

    if 'Weight' not in df.columns:
        print("ОШИБКА: Не найдена целевая переменная 'Weight'")
        print("Доступные колонки:", df.columns.tolist())
        raise ValueError("Целевая переменная 'Weight' не найдена")

    # Сохраняем информацию о доступных колонках
    columns_info_path = os.path.join(output_results_dir, f"columns_availability_{timestamp}.txt")
    with open(columns_info_path, 'w', encoding='utf-8') as f:
        f.write("Доступность колонок (основные для моделирования):\n")
        f.write("="*50 + "\n")
        for col in required_columns:
            f.write(f"  \u2713 {col} {'(доступна)' if col in df.columns else '(отсутствует)'}\n")

    # 2. Очистка данных и подготовка признаков
    print("\nОчистка данных и подготовка признаков:")

    # Define columns to drop
    cols_to_drop_final = ['№', '№ чипа', '№ пробирки', 'Длина гол., см', 'Кол-во лучей в СП', 'Кол-во лучей в АП']

    for col in cols_to_drop_final:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"   Колонка '{col}' удалена.")

    # Explicitly convert potential categorical columns to string type to prevent mixed type errors.
    # These are typically object type columns that are not numeric and not dropped.
    categorical_candidates = []
    for col in df.columns:
        if df[col].dtype == 'object':
            categorical_candidates.append(col)

    for col in categorical_candidates:
        df[col] = df[col].astype(str)
        print(f"   Колонка '{col}' приведена к строковому типу.")

    # Convert all numeric-like columns to numeric, coercing errors
    # Exclude 'Weight' as it's the target.
    numeric_features_raw = [
        'Length', 'Commercial_Length', 'Girth', 'Max_Height', 'Age'
    ]

    for col in numeric_features_raw:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if col not in available_columns: # Ensure print only if not already printed
                print(f"   Колонка '{col}' приведена к числовому типу.")


    # Удаляем строки с пропущенными значениями
    initial_size = df.shape[0]
    df.dropna(inplace=True) # Apply dropna after all coercions, to handle NaNs introduced by coerce
    missing_removed = initial_size - df.shape[0]
    print(f"Удалено строк с пропусками: {missing_removed}")

    # Удаляем некорректные значения (values <= 0 for numeric features that should be positive)
    invalid_removed = 0
    current_numeric_cols_for_positive_check = [col for col in numeric_features_raw if col in df.columns]
    for col in current_numeric_cols_for_positive_check:
        before = df.shape[0]
        df = df[df[col] > 0] # Keep only positive values
        invalid_removed += (before - df.shape[0])
    print(f"Удалено некорректных значений (нулевые или отрицательные) после преобразования: {invalid_removed}")

    # Сохраняем статистику очистки
    cleaning_stats_path = os.path.join(output_results_dir, f"data_cleaning_stats_{timestamp}.txt")
    with open(cleaning_stats_path, 'w', encoding='utf-8') as f:
        f.write("Статистика очистки данных:\n")
        f.write("="*50 + "\n")
        f.write(f"Исходный размер: {initial_size} строк\n")
        f.write(f"Удалено строк с пропусками: {missing_removed}\n")
        f.write(f"Удалено некорректных значений: {invalid_removed}\n")
        f.write(f"Итоговый размер: {df.shape[0]} строк\n")
        f.write(f"Удалено всего: {initial_size - df.shape[0]} строк ({((initial_size - df.shape[0])/initial_size*100):.1f}%)\n")

    # Сохраняем очищенные данные
    cleaned_data_path = os.path.join(output_results_dir, f"cleaned_data_{timestamp}.xlsx")
    df.to_excel(cleaned_data_path, index=False)
    print(f"\u2713 Очищенные данные сохранены: {cleaned_data_path}")

    # df_original now refers to the DataFrame after renaming and dropping specified columns
    df_original = df.copy()

    # 3. Feature engineering
    print("\nСоздание новых признаков:")

    created_features = []

    if all(col in df.columns for col in ['Length', 'Girth', 'Max_Height']):
        df['Volume_Estimate'] = df['Length'] * df['Girth'] * df['Max_Height']
        created_features.append('Volume_Estimate')
        print("\u2713 Volume_Estimate создан")

    if all(col in df.columns for col in ['Length', 'Girth']):
        df['Length_Girth_Ratio'] = df['Length'] / df['Girth']
        created_features.append('Length_Girth_Ratio')
        print("\u2713 Length_Girth_Ratio создан")

    if 'Girth' in df.columns:
        df['Cross_Section_Area'] = np.pi * (df['Girth'] / (2 * np.pi)) ** 2
        created_features.append('Cross_Section_Area')
        print("\u2713 Cross_Section_Area создан")

    features_info_path = os.path.join(output_results_dir, f"created_features_{timestamp}.txt")
    with open(features_info_path, 'w', encoding='utf-8') as f:
        f.write("Созданные признаки:\n")
        f.write("="*50 + "\n")
        for feature in created_features:
            f.write(f"\u2713 {feature}\n")
            if feature == 'Volume_Estimate':
                f.write("  Формула: Length \u00d7 Girth \u00d7 Max_Height\n")
            elif feature == 'Length_Girth_Ratio':
                f.write("  Формула: Length / Girth\n")
            elif feature == 'Cross_Section_Area':
                f.write("  Формула: \u03c0 \u00d7 (Girth/2\u03c0)\u00b2\n")

    # 4. Разделение на признаки и целевую переменную
    X = df.drop('Weight', axis=1)
    y = df['Weight']

    print(f"\nРазмер X: {X.shape}")
    print(f"Размер y: {y.shape}")

    # 5. Определяем типы признаков
    numeric_features = []
    categorical_features = []

    for col in X.columns:
        if pd.api.types.is_numeric_dtype(X[col]):
            numeric_features.append(col)
        else:
            categorical_features.append(col)

    print(f"\nЧисловые признаки ({len(numeric_features)}): {numeric_features}")
    print(f"Категориальные признаки ({len(categorical_features)}): {categorical_features}")

    features_types_path = os.path.join(output_results_dir, f"features_types_{timestamp}.txt")
    with open(features_types_path, 'w', encoding='utf-8') as f:
        f.write("Типы признаков:\n")
        f.write("="*50 + "\n")
        for feat in numeric_features:
            f.write(f"  \u2713 {feat}\n")
        f.write(f"\nКатегориальные признаки ({len(categorical_features)}):\n")
        for feat in categorical_features:
            f.write(f"  \u2713 {feat}\n")

    # 6. Создаем пайплайн предобработки
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features)
            ])

    return X, y, preprocessor, categorical_features, numeric_features, df_original

X, y, preprocessor, categorical_features, numeric_features, df_original_cleaned = preprocess_data(df)

# ======================
# DATA SPLITTING
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"\n{'='*60}")
print("РАЗДЕЛЕНИЕ ДАННЫХ")
print('='*60)
print(f"Обучающая выборка: {X_train.shape[0]} образцов")
print(f"Тестовая выборка: {X_test.shape[0]} образцов")
print(f"Всего признаков: {X_train.shape[1]}")
print(f"\nРаспределение целевой переменной:")
print(f"Минимальный вес: {y.min():.1f} г")
print(f"Максимальный вес: {y.max():.1f} г")
print(f"Средний вес: {y.mean():.1f} г")
print(f"Стандартное отклонение: {y.std():.1f} г")

# Сохраняем информацию о разделении данных
split_info_path = os.path.join(output_results_dir, f"data_split_info_{timestamp}.txt")
with open(split_info_path, 'w', encoding='utf-8') as f:
    f.write("Разделение данных:\n")
    f.write("="*50 + "\n")
    f.write(f"Обучающая выборка: {X_train.shape[0]} образцов ({X_train.shape[0]/len(X)*100:.1f}%)\n")
    f.write(f"Тестовая выборка: {X_test.shape[0]} образцов ({X_test.shape[0]/len(X)*100:.1f}%)\n")
    f.write(f"Всего признаков: {X_train.shape[1]}\n\n")
    f.write("Распределение целевой переменной:\n")
    f.write(f"  Минимальный вес: {y.min():.1f} г\n")
    f.write(f"  Максимальный вес: {y.max():.1f} г\n")
    f.write(f"  Средний вес: {y.mean():.1f} г\n")
    f.write(f"  Медиана: {y.median():.1f} г\n")
    f.write(f"  Стандартное отклонение: {y.std():.1f} г\n")

# ======================
# MODEL TRAINING & EVALUATION (Default Models)
# ======================
print(f"\n{'='*60}")
print("ОБУЧЕНИЕ И ОЦЕНКА МОДЕЛЕЙ (По умолчанию)")
print('='*60)

# Original models dictionary (without Lasso standalone as per instruction)
models = {
    "Линейная регрессия": LinearRegression(),
    "Решающее дерево": DecisionTreeRegressor(random_state=42, max_depth=5),
    "Случайный лес": RandomForestRegressor(random_state=42, n_estimators=100),
    "Градиентный бустинг": GradientBoostingRegressor(random_state=42, n_estimators=100),
    "Голосование регрессоров": VotingRegressor(
        estimators=[
            ('lasso', LassoCV(random_state=42)),
            ('ridge', GradientBoostingRegressor(n_estimators=40, random_state=42)),
            ('knn', KNeighborsRegressor()),
            ('et', ExtraTreesRegressor(n_estimators=40, max_depth=3, random_state=42)),
            ('rf', RandomForestRegressor(n_estimators=40, max_depth=3, random_state=42))
        ]
    )
}

results = {
    "Модель": [],
    "Train RMSE": [],
    "Test RMSE": [],
    "Train MAE": [],
    "Test MAE": [],
    "Train R\u00b2": [],
    "Test R\u00b2": [],
    "CV R\u00b2 Mean": [],
    "CV R\u00b2 Std": [],
    "Status": []
}

# Create figure for all prediction plots (default models)
num_default_models = len(models)
num_rows_default_pred = (num_default_models + 1) // 2
fig_predictions, axes_predictions = plt.subplots(num_rows_default_pred, 2, figsize=(14, 5 * num_rows_default_pred))
axes_predictions = axes_predictions.flatten()

# Dictionary to store predictions of all models
all_predictions = {}

for idx, (name, model) in enumerate(models.items()):
    print(f"\nОбучение модели: {name}")

    # Создаем пайплайн
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Обучение модели
    pipeline.fit(X_train, y_train)

    # Предсказания
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # Сохраняем предсказания
    all_predictions[name] = {
        'train_pred': y_train_pred,
        'test_pred': y_test_pred,
        'model': pipeline
    }

    # Расчет метрик
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Кросс-валидация
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')

    # Сохраняем результаты
    results["Модель"].append(name)
    results["Train RMSE"].append(train_rmse)
    results["Test RMSE"].append(test_rmse)
    results["Train MAE"].append(train_mae)
    results["Test MAE"].append(test_mae)
    results["Test R\u00b2"].append(test_r2)
    results["Train R\u00b2"].append(train_r2)
    results["CV R\u00b2 Mean"].append(np.mean(cv_scores))
    results["CV R\u00b2 Std"].append(np.std(cv_scores))
    results["Status"].append("Default")

    # График предсказаний
    ax = axes_predictions[idx]
    ax.scatter(y_test, y_test_pred, alpha=0.6, s=50)
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    ax.set_xlabel('Фактический вес (г)')
    ax.set_ylabel('Предсказанный вес (г)')
    ax.set_title(f'{name}\nR\u00b2 = {test_r2:.3f}')
    ax.grid(True, alpha=0.3)

    print(f"  Test R\u00b2: {test_r2:.3f}")
    print(f"  Test RMSE: {test_rmse:.1f} г")
    print(f"  Test MAE: {test_mae:.1f} г")

plt.tight_layout()
predictions_plot_path = os.path.join(output_results_dir, f"predictions_comparison_default_{timestamp}.png")
plt.savefig(predictions_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"\u2713 График предсказаний сохранен: {predictions_plot_path}")

# Создаем DataFrame с результатами
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ МОДЕЛЕЙ (По умолчанию)")
print('='*60)
print(results_df.sort_values(by='Test MAE', ascending=True).to_string(index=False))

# Save initial results models
models_results_path_default = os.path.join(output_results_dir, f"models_results_default_{timestamp}.xlsx")
results_df.to_excel(models_results_path_default, index=False)
print(f"\u2713 Результаты моделей (по умолчанию) сохранены: {models_results_path_default}")

# ======================
# HYPERPARAMETER TUNING (All Models)
# ======================
print(f"\n{'='*60}")
print("НАСТРОЙКА ГИПЕРПАРАМЕТРОВ (Все модели)")
print('='*60)

# Define expanded hyperparameter grids for tuning
param_grids = {
    "Линейная регрессия": {
        'model__fit_intercept': [True, False]
    },
    "Решающее дерево": {
        'model__max_depth': [3, 5, 7, 10, 15, None],
        'model__min_samples_split': [2, 5, 10, 20],
        'model__min_samples_leaf': [1, 2, 4, 8],
        'model__max_features': ['sqrt', 'log2', None]
    },
    "Случайный лес": {
        'model__n_estimators': [50, 100, 200, 300],
        'model__max_depth': [5, 10, 20, 30, None],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 'log2', 0.8] # Use float for max_features as an example
    },
    "Градиентный бустинг": {
        'model__n_estimators': [50, 100, 200, 300],
        'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
        'model__max_depth': [3, 5, 7, 9],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4]
    },
    "Голосование регрессоров": {
        'model__lasso__eps': [1e-4, 1e-3],
        'model__lasso__n_alphas': [100],
        'model__ridge__n_estimators': [50, 100],
        'model__ridge__learning_rate': [0.05, 0.1],
        'model__ridge__max_depth': [3, 5],
        'model__knn__n_neighbors': [3, 5, 7],
        'model__knn__weights': ['uniform', 'distance'],
        'model__et__n_estimators': [50, 100],
        'model__et__max_depth': [5, 10],
        'model__rf__n_estimators': [50, 100],
        'model__rf__max_depth': [5, 10],
        'model__weights': [
            (0.2,0.2,0.2,0.2,0.2), # Equal weights
            (0.1,0.2,0.2,0.25,0.25),
            (0.2,0.1,0.15,0.25,0.3)
        ]
    }
}

# Dictionaries to store tuned models and their results
tuned_results = {
    "Модель": [],
    "Train RMSE": [],
    "Test RMSE": [],
    "Train MAE": [],
    "Test MAE": [],
    "Train R\u00b2": [],
    "Test R\u00b2": [],
    "CV R\u00b2 Mean": [],
    "CV R\u00b2 Std": [],
    "Status": [] # To indicate if it's default or tuned
}
tuned_all_predictions = {}
tuned_models = {}

for name, model_instance in models.items():
    if name in param_grids:
        print(f"\nЗапуск GridSearchCV для модели: {name}")

        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model_instance)
        ])

        # GridSearchCV for the current model
        try:
            grid_search = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grids[name],
                cv=3, # Reduced cv for faster execution in this example
                scoring='r2',
                n_jobs=-1,
                verbose=1
            )
            grid_search.fit(X_train, y_train)

            best_pipeline = grid_search.best_estimator_
            tuned_models[name] = best_pipeline

            # Predictions from tuned model
            y_train_pred_tuned = best_pipeline.predict(X_train)
            y_test_pred_tuned = best_pipeline.predict(X_test)

            # Store predictions
            tuned_all_predictions[f"{name}_настроенная"] = {
                'train_pred': y_train_pred_tuned,
                'test_pred': y_test_pred_tuned,
                'model': best_pipeline
            }

            # Calculate metrics for tuned model
            train_rmse_tuned = np.sqrt(mean_squared_error(y_train, y_train_pred_tuned))
            test_rmse_tuned = np.sqrt(mean_squared_error(y_test, y_test_pred_tuned))
            train_mae_tuned = mean_absolute_error(y_train, y_train_pred_tuned)
            test_mae_tuned = mean_absolute_error(y_test, y_test_pred_tuned)
            train_r2_tuned = r2_score(y_train, y_train_pred_tuned)
            test_r2_tuned = r2_score(y_test, y_test_pred_tuned)

            # Cross-validation score for the best estimator from GridSearchCV
            cv_scores_tuned_mean = grid_search.best_score_
            # Calculate std of test scores across folds for the best parameters
            best_params_index = np.where(grid_search.cv_results_['rank_test_score'] == 1)[0][0]
            cv_scores_tuned_std = grid_search.cv_results_['std_test_score'][best_params_index]

            # Store tuned results
            tuned_results["Модель"].append(f"{name}_настроенная")
            tuned_results["Train RMSE"].append(train_rmse_tuned)
            tuned_results["Test RMSE"].append(test_rmse_tuned)
            tuned_results["Train MAE"].append(train_mae_tuned)
            tuned_results["Test MAE"].append(test_mae_tuned)
            tuned_results["Test R\u00b2"].append(test_r2_tuned)
            tuned_results["Train R\u00b2"].append(train_r2_tuned)
            tuned_results["CV R\u00b2 Mean"].append(cv_scores_tuned_mean)
            tuned_results["CV R\u00b2 Std"].append(cv_scores_tuned_std)
            tuned_results["Status"].append("Tuned")

            print(f"  Лучший R\u00b2 (CV): {grid_search.best_score_:.3f}")
            print(f"  Лучшие параметры: {grid_search.best_params_}")
            print(f"  Test R\u00b2 (настроенная): {test_r2_tuned:.3f}")

            # Save tuning results to file
            tuning_results_path = os.path.join(output_results_dir, f"tuning_results_{name.replace(' ', '_')}_{timestamp}.txt")
            with open(tuning_results_path, 'w', encoding='utf-8') as f:
                f.write(f"Настройка гиперпараметров для модели: {name}\n")
                f.write("="*50 + "\n")
                f.write(f"Лучший R\u00b2 (CV): {grid_search.best_score_:.3f}\n")
                f.write("Лучшие параметры:\n")
                for param, value in grid_search.best_params_.items():
                    f.write(f"  {param}: {value}\n")
                f.write(f"Test R\u00b2 (настроенная): {test_r2_tuned:.3f}\n")
            print(f"\u2713 Результаты настройки для '{name}' сохранены: {tuning_results_path}")

        except Exception as e:
            print(f"  Ошибка при настройке гиперпараметров для '{name}': {e}")
            # Append placeholder results if tuning fails
            tuned_results["Модель"].append(f"{name}_настроенная")
            tuned_results["Train RMSE"].append(np.nan)
            tuned_results["Test RMSE"].append(np.nan)
            tuned_results["Train MAE"].append(np.nan)
            tuned_results["Test MAE"].append(np.nan)
            tuned_results["Test R\u00b2"].append(np.nan)
            tuned_results["Train R\u00b2"].append(np.nan)
            tuned_results["CV R\u00b2 Mean"].append(np.nan)
            tuned_results["CV R\u00b2 Std"].append(np.nan)
            tuned_results["Status"].append("Failed Tuning")
    else:
        print(f"  Параметры для настройки модели '{name}' не определены.")


# ======================
# CONSOLIDATE AND REPORT ALL RESULTS
# ======================
print(f"\n{'='*60}")
print("КОНСОЛИДАЦИЯ И ОТЧЕТ ВСЕХ РЕЗУЛЬТАТОВ")
print('='*60)

# Convert default results to DataFrame and add 'Status'
results_df['Status'] = 'Default'

# Convert tuned results to DataFrame
tuned_results_df = pd.DataFrame(tuned_results)

# Combine default and tuned results
all_models_results_df = pd.concat([results_df, tuned_results_df], ignore_index=True)

# Sort by Test MAE for better comparison
all_models_results_df_sorted = all_models_results_df.sort_values(by='Test MAE', ascending=True)

print(f"\n{'='*60}")
print("СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ (ПО УМОЛЧАНИЮ И НАСТРОЕННЫХ)")
print('='*60)
print(all_models_results_df_sorted.to_string(index=False))

# Save all models results
all_models_results_path = os.path.join(output_results_dir, f"all_models_results_{timestamp}.xlsx")
all_models_results_df.to_excel(all_models_results_path, index=False)
print(f"\u2713 Все результаты моделей (по умолчанию и настроенных) сохранены: {all_models_results_path}")


# Update all_predictions with tuned predictions
all_predictions.update(tuned_all_predictions)

# Now update the sections that use 'all_predictions' and 'results_df'
# ======================
# CREATE PREDICTIONS TABLE (UPDATED)
# ======================
print(f"\n{'='*60}")
print("СОХРАНЕНИЕ ПРЕДСКАЗАННЫХ РЕЗУЛЬТАТОВ (ОБНОВЛЕНО)")
print('='*60)

# Create a DataFrame with original data for the test set
test_indices = X_test.index
predictions_df = pd.DataFrame()

df_raw_test_subset = df_raw_initial.loc[test_indices]
for col in df_raw_test_subset.columns:
    if col == 'Вес, г':
        predictions_df[f'Оригинал_Вес_г'] = df_raw_test_subset[col]
    elif col == 'Возраст, л (по состоянию на весну 2025)':
        predictions_df[f'Оригинал_Возраст'] = df_raw_test_subset[col]
    else:
        predictions_df[f'Оригинал_{col}'] = df_raw_test_subset[col]

predictions_df['Фактический_Вес_г'] = y_test

# Add predictions from all models (default and tuned)
for model_name, pred_data in all_predictions.items():
    predictions_df[f'Предсказание_{model_name}'] = pred_data['test_pred']
    predictions_df[f'Ошибка_{model_name}'] = y_test - pred_data['test_pred']
    predictions_df[f'Относительная_ошибка_{model_name}'] = np.abs(y_test - pred_data['test_pred']) / y_test * 100

# Add additional calculations
prediction_columns = [col for col in predictions_df.columns if col.startswith('Предсказание_')]
if prediction_columns:
    predictions_df['Среднее_предсказание'] = predictions_df[prediction_columns].mean(axis=1)
    predictions_df['Медиана_предсказаний'] = predictions_df[prediction_columns].median(axis=1)
    predictions_df['Std_предсказаний'] = predictions_df[prediction_columns].std(axis=1)

predictions_df = predictions_df.sort_values('Фактический_Вес_г')

# Determine the best model (now from all_models_results_df_sorted)
best_model_overall_name = all_models_results_df_sorted.iloc[0]['Модель']
print(f"Лучшая модель (по MAE, среди всех): {best_model_overall_name}")

# Create summary table by species (if 'Species' column exists)
if 'Оригинал_Вид' in predictions_df.columns:
    species_summary = predictions_df.groupby('Оригинал_Вид').agg({
        'Фактический_Вес_г': ['count', 'mean', 'std', 'min', 'max'],
        f'Предсказание_{best_model_overall_name}': ['mean', 'std'],
        f'Относительная_ошибка_{best_model_overall_name}': 'mean'
    }).round(2)
    print(f"\nСводная статистика по видам рыб (на основе лучшей модели '{best_model_overall_name}'):")
    print(species_summary)
    species_summary_path = os.path.join(output_results_dir, f"species_summary_all_models_{timestamp}.xlsx")
    species_summary.to_excel(species_summary_path)
    print(f"\u2713 Сводная статистика по видам сохранена: {species_summary_path}")

# Save to Excel file
output_filename = os.path.join(output_results_dir, f"fish_weight_predictions_all_models_{timestamp}.xlsx")

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    predictions_df.to_excel(writer, sheet_name='Все_предсказания', index=True)
    all_models_results_df_sorted.to_excel(writer, sheet_name='Сводка_всех_моделей', index=False)

    num_predictions_to_show = min(10, len(predictions_df))
    if f'Относительная_ошибка_{best_model_overall_name}' in predictions_df.columns:
        best_predictions = predictions_df.nsmallest(num_predictions_to_show, f'Относительная_ошибка_{best_model_overall_name}')
        worst_predictions = predictions_df.nlargest(num_predictions_to_show, f'Относительная_ошибка_{best_model_overall_name}')
        best_predictions.to_excel(writer, sheet_name='Лучшие_предсказания', index=True)
        worst_predictions.to_excel(writer, sheet_name='Худшие_предсказания', index=True)
    else:
        print(f"Warning: Relative error for best model '{best_model_overall_name}' not found for best/worst predictions.")

    errors_summary = pd.DataFrame()
    for model_name in all_predictions.keys():
        error_col = f'Ошибка_{model_name}'
        rel_error_col = f'Относительная_ошибка_{model_name}'
        if error_col in predictions_df.columns and rel_error_col in predictions_df.columns:
            model_errors = {
                'Модель': model_name,
                'Средняя_абсолютная_ошибка_г': predictions_df[error_col].abs().mean(),
                'Средняя_относительная_ошибка_%': predictions_df[rel_error_col].mean(),
                'Максимальная_ошибка_г': predictions_df[error_col].abs().max(),
                'Точность_до_10%_%': (predictions_df[rel_error_col] <= 10).sum() / len(predictions_df) * 100,
                'Точность_до_5%_%': (predictions_df[rel_error_col] <= 5).sum() / len(predictions_df) * 100
            }
            errors_summary = pd.concat([errors_summary, pd.DataFrame([model_errors])], ignore_index=True)
    if not errors_summary.empty:
        errors_summary.to_excel(writer, sheet_name='Статистика_ошибок', index=False)
    else:
        print("Warning: No error statistics generated (possibly due to missing error columns).")


print(f"\n\u2713 Результаты сохранены в файл: {output_filename}")
print(f"\u2713 Всего сохранено записей: {len(predictions_df)}")
print(f"\u2713 Файл содержит листы: Все_предсказания, Сводка_всех_моделей, Лучшие_предсказания, Худшие_предсказания, Статистика_ошибок")

# ======================
# FEATURE IMPORTANCE ANALYSIS (UPDATED)
# ======================
print(f"\n{'='*60}")
print("АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ (ОБНОВЛЕНО)")
print('='*60)

# Collect all tree-based models, including tuned ones
all_tree_models = {
    "Решающее дерево": models["Решающее дерево"],
    "Случайный лес": models["Случайный лес"],
    "Градиентный бустинг": models["Градиентный бустинг"],
}
all_tree_models.update({
    f"{name}_настроенная": tuned_models[name]
    for name in tuned_models if name in ["Решающее дерево", "Случайный лес", "Градиентный бустинг"]
})

num_tree_models = len(all_tree_models)
if num_tree_models > 0:
    fig_importance, axes_importance = plt.subplots(1, num_tree_models, figsize=(6 * num_tree_models, 6))
    axes_importance = axes_importance.flatten() if num_tree_models > 1 else [axes_importance]
else:
    print("Нет древовидных моделей для анализа важности признаков.")
    fig_importance = None

if fig_importance is not None:
    for idx, (name, pipeline_or_model) in enumerate(all_tree_models.items()):
        # If it's a tuned model, it's already a pipeline, otherwise wrap it
        if isinstance(pipeline_or_model, Pipeline):
            pipeline = pipeline_or_model
            model = pipeline.named_steps['model']
        else: # It's a default model instance, create a pipeline
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('model', pipeline_or_model)
            ])
            # We must fit the pipeline to ensure the preprocessor is trained
            # and feature names can be retrieved correctly. It's safe to re-fit
            # if it's a new pipeline instance.
            pipeline.fit(X_train, y_train)
            model = pipeline.named_steps['model']

        if hasattr(model, 'feature_importances_'):
            try:
                feature_names = numeric_features.copy()
                if categorical_features:
                    # Need to fit the preprocessor to get category names
                    fitted_preprocessor = pipeline.named_steps['preprocessor'].fit(X_train)
                    cat_features_ohe = fitted_preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
                    feature_names.extend(cat_features_ohe)

                importances = model.feature_importances_

                # Ensure feature_names and importances have the same length
                if len(feature_names) == len(importances):
                    importance_df = pd.DataFrame({
                        'Признак': feature_names,
                        'Важность': importances
                    }).sort_values('Важность', ascending=True)

                    importance_filename = os.path.join(output_results_dir, f"feature_importance_{name.replace(' ', '_')}_{timestamp}.xlsx")
                    importance_df.to_excel(importance_filename, index=False)
                    print(f"\u2713 Важность признаков для модели '{name}' сохранена в файл: {importance_filename}")

                    ax = axes_importance[idx]
                    bars = ax.barh(importance_df['Признак'], importance_df['Важность'])
                    ax.set_xlabel('Важность')
                    ax.set_title(f'{name}\nОбщая важность: {importances.sum():.3f}')

                    for i, (bar, importance) in enumerate(zip(bars, importance_df['Важность'])):
                        if importance > 0.1:
                            bar.set_color('darkred')
                        elif importance > 0.05:
                            bar.set_color('orange')
                        else:
                            bar.set_color('lightblue')
                else:
                    print(f"Warning: Mismatch between feature names length ({len(feature_names)}) and importances length ({len(importances)}) for {name}.")

            except Exception as e:
                print(f"Ошибка при построении важности признаков для {name}: {e}")
        else:
            print(f"Модель '{name}' не имеет атрибута 'feature_importances_'.")

    plt.tight_layout()
    importance_plot_path = os.path.join(output_results_dir, f"feature_importance_comparison_all_models_{timestamp}.png")
    plt.savefig(importance_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\u2713 График важности признаков сохранен: {importance_plot_path}")


# ======================
# RESIDUAL ANALYSIS (UPDATED)
# ======================
print(f"\n{'='*60}")
print("АНАЛИЗ ОСТАТКОВ (ОБНОВЛЕНО)")
print('='*60)

num_models_residuals = len(all_predictions) # Use all_predictions to include tuned models
num_rows_residuals = (num_models_residuals + 1) // 2
fig_residuals, axes_residuals = plt.subplots(num_rows_residuals, 2, figsize=(14, 5 * num_rows_residuals))
axes_residuals = axes_residuals.flatten()

for idx, (name, pred_data) in enumerate(all_predictions.items()):
    y_pred = pred_data['test_pred']
    residuals = y_test - y_pred

    ax = axes_residuals[idx]
    ax.scatter(y_pred, residuals, alpha=0.6)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax.set_xlabel('Предсказанные значения (г)')
    ax.set_ylabel('Остатки (г)')
    ax.set_title(f'{name}\nСреднее остатков: {residuals.mean():.1f} г')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
residuals_plot_path = os.path.join(output_results_dir, f"residuals_analysis_all_models_{timestamp}.png")
plt.savefig(residuals_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"\u2713 График анализа остатков сохранен: {residuals_plot_path}")

# ======================
# ERROR DISTRIBUTION (UPDATED)
# ======================
print(f"\n{'='*60}")
print("РАСПРЕДЕЛЕНИЕ ОШИБОК (ОБНОВЛЕНО)")
print('='*60)

num_models_errors = len(all_predictions) # Use all_predictions to include tuned models
num_rows_errors = (num_models_errors + 1) // 2
fig_errors, axes_errors = plt.subplots(num_rows_errors, 2, figsize=(14, 5 * num_rows_errors))
axes_errors = axes_errors.flatten()

for idx, (name, pred_data) in enumerate(all_predictions.items()):
    y_pred = pred_data['test_pred']
    errors = y_test - y_pred

    ax = axes_errors[idx]
    sns.histplot(errors, kde=True, ax=ax, bins=20)
    ax.axvline(x=0, color='r', linestyle='--', linewidth=2)
    ax.set_xlabel('Ошибка предсказания (г)')
    ax.set_ylabel('Частота')
    ax.set_title(f'{name}\nСреднее: {errors.mean():.1f} г, STD: {errors.std():.1f} г')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
errors_plot_path = os.path.join(output_results_dir, f"error_distribution_all_models_{timestamp}.png")
plt.savefig(errors_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"\u2713 График распределения ошибок сохранен: {errors_plot_path}")

# ======================
# FINAL SUMMARY (UPDATED)
# ======================
print(f"\n{'='*60}")
print("ИТОГОВЫЙ ОТЧЕТ (ОБНОВЛЕНО)")
print('='*60)

# Determine the best overall model from all_models_results_df_sorted
best_model_overall_name = all_models_results_df_sorted.iloc[0]['Модель']
best_model_overall_metrics = all_models_results_df_sorted.iloc[0]

print(f"Источник данных: {file_path}")
print(f"Всего записей: {df.shape[0]}")
print(f"Использовано признаков: {X.shape[1]}")
print(f"Целевая переменная: Вес рыбы (г)")
print(f"\nЛучшая модель (среди всех): {best_model_overall_name}")
print(f"  R\u00b2 на тесте: {best_model_overall_metrics['Test R\u00b2']:.3f}")
print(f"  RMSE на тесте: {best_model_overall_metrics['Test RMSE']:.1f} г")
print(f"  MAE на тесте: {best_model_overall_metrics['Test MAE']:.1f} г")
print(f"\nРаспределение веса:")
print(f"  Минимальный: {y.min():.1f} г")
print(f"  Максимальный: {y.max():.1f} г")
print(f"  Средний: {y.mean():.1f} г")
print(f"  Стандартное отклонение: {y.std():.1f} г")

print(f"\n\uD83D\uDCC1 СОХРАНЕННЫЕ ФАЙЛЫ в Google Drive:")
print(f"  Основная папка: {drive_results_dir}")
print(f"  Папка текущего анализа: {output_results_dir}")
print(f"\n  ФАЙЛЫ АНАЛИЗА:")
print(f"  1. fish_weight_predictions_all_models_{timestamp}.xlsx - Основные предсказания (5 листов, включая настроенные)")
for model_name in all_tree_models.keys():
    print(f"  2. feature_importance_{model_name.replace(' ', '_')}_{timestamp}.xlsx")
for model_name in tuned_models.keys():
    print(f"  3. tuning_results_{model_name.replace(' ', '_')}_{timestamp}.txt - Отчет по настройке '{model_name}'")
print(f"  4. all_models_results_{timestamp}.xlsx - Результаты всех моделей (по умолчанию и настроенных)")
if 'Оригинал_Вид' in predictions_df.columns:
    print(f"  5. species_summary_all_models_{timestamp}.xlsx - Сводная статистика по видам (на основе лучшей модели)")
print(f"\n  ГРАФИКИ:")
print(f"  6. predictions_comparison_default_{timestamp}.png - Графики сравнения предсказаний (по умолчанию)") # Keep original if desired
print(f"  7. feature_importance_comparison_all_models_{timestamp}.png - Графики важности признаков (все модели)")
print(f"  8. residuals_analysis_all_models_{timestamp}.png - Графики анализа остатков (все модели)")
print(f"  9. error_distribution_all_models_{timestamp}.png - Графики распределения ошибок (все модели)")
print(f"  10. final_model_comparison_mae_all_models_{timestamp}.png - Финальный график сравнения всех моделей (по MAE)")

print(f"\n  ВСПОМОГАТЕЛЬНЫЕ ФАЙЛЫ (без изменений):")
print(f"  11. cleaned_data_{timestamp}.xlsx")
print(f"  12. data_info_{timestamp}.txt")
print(f"  13. data_split_info_{timestamp}.txt")
print(f"  14. data_cleaning_stats_{timestamp}.txt")
print(f"  15. created_features_{timestamp}.txt")
print(f"  16. source_file_info_{timestamp}.txt")
print(f"  17. analysis_report_{timestamp}.txt (этот файл)")


# Create final model comparison plot (by MAE for all models)
plt.figure(figsize=(12, 8))
models_sorted_for_plot = all_models_results_df_sorted.copy()
models_sorted_for_plot['Model_Label'] = models_sorted_for_plot['Модель'] + ' (' + models_sorted_for_plot['Status'] + ')'

bars = plt.barh(models_sorted_for_plot['Model_Label'], models_sorted_for_plot['Test MAE'], color=['skyblue' if s == 'Default' else 'lightcoral' for s in models_sorted_for_plot['Status']])
plt.xlabel('MAE Score (г)')
plt.title('Сравнение производительности всех моделей (Test MAE)')
plt.xlim(left=0)

for i, (bar, mae) in enumerate(zip(bars, models_sorted_for_plot['Test MAE'])):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{mae:.1f}', va='center')

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
final_plot_path = os.path.join(output_results_dir, f"final_model_comparison_mae_all_models_{timestamp}.png")
plt.savefig(final_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"\u2713 Финальный график сравнения всех моделей (по MAE) сохранен: {final_plot_path}")

# Update and save the final report
report_filename = os.path.join(output_results_dir, f"analysis_report_{timestamp}.txt")
with open(report_filename, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("ИТОГОВЫЙ ОТЧЕТ АНАЛИЗА ВЕСА РЫБ (ОБНОВЛЕНО)\n")
    f.write("="*60 + "\n\n")
    f.write(f"Дата анализа: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Источник данных: {file_path}\n")
    f.write(f"Всего записей: {df.shape[0]}\n")
    f.write(f"Использовано признаков: {X.shape[1]}\n")
    f.write(f"Целевая переменная: Вес рыбы (г)\n\n")
    f.write(f"Лучшая модель (среди всех): {best_model_overall_name}\n")
    f.write(f"  R\u00b2 на тесте: {best_model_overall_metrics['Test R\u00b2']:.3f}\n")
    f.write(f"  RMSE на тесте: {best_model_overall_metrics['Test RMSE']:.1f} г\n")
    f.write(f"  MAE на тесте: {best_model_overall_metrics['Test MAE']:.1f} г\n\n")
    f.write("Распределение веса:\n")
    f.write(f"  Минимальный: {y.min():.1f} г\n")
    f.write(f"  Максимальный: {y.max():.1f} г\n")
    f.write(f"  Средний: {y.mean():.1f} г\n")
    f.write(f"  Стандартное отклонение: {y.std():.1f} г\n\n")
    f.write("="*60 + "\n")
    f.write("СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ (ПО УМОЛЧАНИЮ И НАСТРОЕННЫХ):\n")
    f.write("="*60 + "\n")
    f.write(all_models_results_df_sorted.to_string(index=False))
    f.write("\n\n")
    f.write("="*60 + "\n")
    f.write("СОХРАНЕННЫЕ ФАЙЛЫ:\n")
    f.write("="*60 + "\n")
    f.write(f"Основная папка: {drive_results_dir}\n")
    f.write(f"Папка текущего анализа: {output_results_dir}\n\n")
    f.write("ФАЙЛЫ АНАЛИЗА:\n")
    f.write(f"1. fish_weight_predictions_all_models_{timestamp}.xlsx - Основные предсказания (5 листов, включая настроенные)\n")
    for model_name in all_tree_models.keys():
        f.write(f"2. feature_importance_{model_name.replace(' ', '_')}_{timestamp}.xlsx\n")
    for model_name in tuned_models.keys():
        f.write(f"3. tuning_results_{model_name.replace(' ', '_')}_{timestamp}.txt - Отчет по настройке '{model_name}'\n")
    f.write(f"4. all_models_results_{timestamp}.xlsx - Результаты всех моделей (по умолчанию и настроенных)\n")
    if 'Оригинал_Вид' in predictions_df.columns:
        f.write(f"5. species_summary_all_models_{timestamp}.xlsx - Сводная статистика по видам (на основе лучшей модели)\n")
    f.write("\nГРАФИКИ:\n")
    f.write(f"6. predictions_comparison_default_{timestamp}.png (по умолчанию)\n")
    f.write(f"7. feature_importance_comparison_all_models_{timestamp}.png (все модели)\n")
    f.write(f"8. residuals_analysis_all_models_{timestamp}.png (все модели)\n")
    f.write(f"9. error_distribution_all_models_{timestamp}.png (все модели)\n")
    f.write(f"10. final_model_comparison_mae_all_models_{timestamp}.png (все модели)\n")
    f.write("\nВСПОМОГАТЕЛЬНЫЕ ФАЙЛЫ (без изменений):\n")
    f.write(f"11. cleaned_data_{timestamp}.xlsx\n")
    f.write(f"12. data_info_{timestamp}.txt\n")
    f.write(f"13. data_split_info_{timestamp}.txt\n")
    f.write(f"14. data_cleaning_stats_{timestamp}.txt\n")
    f.write(f"15. created_features_{timestamp}.txt\n")
    f.write(f"16. source_file_info_{timestamp}.txt\n")
    f.write(f"17. analysis_report_{timestamp}.txt (этот файл)\n")

print(f"\u2713 Итоговый отчет обновлен и сохранен: {report_filename}")

# Update README file
readme_path = os.path.join(output_results_dir, "README.txt")
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("ПАПКА С РЕЗУЛЬТАТАМИ АНАЛИЗА ВЕСА РЫБ (ОБНОВЛЕНО)\n")
    f.write("="*60 + "\n\n")
    f.write("Эта папка содержит результаты анализа данных о весе рыб, включая сравнение моделей до и после настройки гиперпараметров.\n\n")
    f.write("ОСНОВНЫЕ ФАЙЛЫ:\n")
    f.write("1. fish_weight_predictions_all_models_XXXXXX.xlsx - Основной файл с предсказаниями (5 листов, включая настроенные)\n")
    f.write("   Содержит листы:\n")
    f.write("   - Все_предсказания: Полные результаты предсказаний всех моделей\n")
    f.write("   - Сводка_всех_моделей: Сравнение производительности всех моделей (по умолчанию и настроенных)\n")
    f.write(f"   - Лучшие_предсказания: {num_predictions_to_show} самых точных предсказаний по лучшей модели\n")
    f.write(f"   - Худшие_предсказания: {num_predictions_to_show} самых неточных предсказаний по лучшей модели\n")
    f.write("   - Статистика_ошибок: Детальная статистика ошибок по всем моделям\n\n")
    f.write("2. analysis_report_XXXXXX.txt - Полный отчет об анализе\n")
    f.write("3. *.png файлы - Графики и визуализации для всех моделей\n")
    f.write("4. *.xlsx файлы - Дополнительные данные и результаты\n")
    f.write("5. *.txt файлы - Вспомогательная информация и логи настройки\n\n")
    f.write("ДАТА АНАЛИЗА: " + datetime.now().strftime('%Y-%m-%d %H:%M:%S') + "\n")

print(f"\n\u2713 Файл README обновлен: {readme_path}")

print(f"\n{'='*60}")
print("АНАЛИЗ ЗАВЕРШЕН УСПЕШНО!")
print('='*60)
print(f"Все файлы с результатами сохранены в Google Drive:")
print(f"Основная папка: {drive_results_dir}")
print(f"Папка текущего анализа: {output_results_dir}")
print(f"Основной файл: fish_weight_predictions_all_models_{timestamp}.xlsx")
print(f"\nДля доступа к файлам перейдите в Google Drive:")
print(f"Мой Диск \u2192 fish_weight_prediction_results \u2192 analysis_{timestamp}")